In [ ]:
import os
import json
import pandas as pd
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# LangChain 관련
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores.faiss import FAISS
from langchain.schema import Document
from langchain.docstore.in_memory import InMemoryDocstore
from langchain.prompts import PromptTemplate
import faiss

# ----------------------------
# 환경설정
# ----------------------------
load_dotenv("env.txt")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ----------------------------
# 감정별 프롬프트
# ----------------------------
sentiment_prompts = {
    "Angry": "사용자가 화가 난 상태입니다. 화난 말투 표현.",
    "Happy": "사용자가 행복한 상태입니다. 행복한 말투 표현.",
    "Sad": "사용자가 슬픈 상태입니다. 슬픈 말투 표현.",
    "Disgust": "사용자가 역겨움/불쾌함을 표현했습니다. 역겨움, 불쾌함 공감.",
    "Neutral": "사용자가 중립적입니다. 일반적인 정보 제공과 자연스러운 대화를 이어가세요.",
    "Surprise": "사용자가 놀람을 표현했습니다. 놀람의 이유를 묻거나 공감하며 대화를 이어가세요.",
    "Fear": "사용자가 두려움을 표현했습니다. 안정감을 주고 안전한 느낌을 전달하세요."
}

# ----------------------------
# 전역 상태
# ----------------------------
history: List[Dict[str, Any]] = []
last_sentiment: Optional[str] = None
important_sentences_rows: List[Dict[str, Any]] = []
sentiment_change_index: Optional[int] = None

CSV_PATH = "important_sentences.csv"
VEC_PATH = "vector_store_faiss"
META_PATH = "vector_store_meta.json"

# CSV 헤더 보장
if not os.path.exists(CSV_PATH):
    pd.DataFrame(columns=["starttime", "text", "sentiment"]).to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

# ----------------------------
# LangChain Embeddings & FAISS VectorStore
# ----------------------------
embeddings_model = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
dim = len(embeddings_model.embed_query("test"))

index = faiss.IndexFlatL2(dim)
vecdb = FAISS(
    index=index,
    embedding_function=embeddings_model,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

# CSV → VectorStore 동기화
def add_to_vecdb(rows: List[Dict[str, Any]]):
    if not rows:
        return
    texts = [f"Text: {r['text']}\nSentiment: {r['sentiment']}" for r in rows]
    docs = [Document(page_content=t, metadata=r) for t, r in zip(texts, rows)]
    vecdb.add_documents(docs)
    vecdb.save_local(VEC_PATH)

def sync_csv_to_vecdb():
    if not os.path.exists(CSV_PATH):
        return
    df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
    if df.empty:
        return
    add_to_vecdb(df.to_dict(orient="records"))

sync_csv_to_vecdb()

# ----------------------------
# GPT 모델 & PromptTemplate
# ----------------------------
chat_model = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name="gpt-4o-mini",
    temperature=0.3
)

prompt_template = PromptTemplate(
    input_variables=["final_prompt"],
    template="""너는 'AI 인형'이라는 가상 캐릭터야.
유저가 제공한 대화 내용과 요약 정보(final_prompt)에 따라 행동해야 해.
행동 지침:
1. 유저의 감정과 상황을 파악하고 공감해줘.
2. 유저의 최근 대화, 중요한 문장, 장기 기억 요약을 참고하여 맥락에 맞게 답변.
3. 최근 대화로 지금 말하고 있는 맥락 파악.
4. 중요한 문장은 유저에 대한 특성 파악.
5. 장기 기억 요약으로 유저에 대한 특별한 상황 인식.
6. 질문, 제안, 조언 등을 적절히 섞어 자연스럽게 대화 이어가기.
7. 1~3문장 정도로 간결하게 작성.

{final_prompt}"""
)

# Runnable 스타일 연결
llm_pipeline = prompt_template | chat_model

# ----------------------------
# 요약 함수
# ----------------------------
def summarize_with_gpt(prompt: str) -> str:
    return llm_pipeline.invoke({"final_prompt": prompt})

def summarize_recent_context_str(text: str) -> str:
    prompt = f"다음 대화 내용을 주요 내용 중심으로 5문장 이하로 요약하세요:\n{text}"
    return summarize_with_gpt(prompt)

def summarize_rag_context_text(rag_text: str) -> str:
    if not rag_text.strip():
        return ""
    combined_text = " ".join(dict.fromkeys([line.strip() for line in rag_text.split("\n") if line.strip()]))
    prompt = f"다음 내용을 주요 내용 중심으로 5문장 이하로 요약하세요:\n{combined_text}"
    return summarize_with_gpt(prompt)

# ----------------------------
# AI 응답 처리
# ----------------------------
def handle_user_input(starttime: str, text: str, sentiment: str) -> str:
    global history, last_sentiment, sentiment_change_index, important_sentences_rows

    importance_type = "normal"
    if last_sentiment != sentiment:
        importance_type = "sentiment_change"
    last_sentiment = sentiment

    entry = {"starttime": starttime, "text": text, "sentiment": sentiment}
    history.append(entry)

    if importance_type == "sentiment_change":
        sentiment_change_index = len(history) - 1

    if sentiment_change_index is not None and len(history) >= sentiment_change_index + 2:
        start_idx = max(0, sentiment_change_index - 2)
        end_idx = min(len(history), sentiment_change_index + 2)
        selected = history[start_idx:end_idx]
        combined_text = " ".join([h["text"] for h in selected])
        combined_row = {
            "starttime": selected[0]["starttime"],
            "text": combined_text,
            "sentiment": history[sentiment_change_index]["sentiment"]
        }
        if not any(row["text"] == combined_text for row in important_sentences_rows):
            important_sentences_rows.append(combined_row)
            pd.DataFrame([combined_row]).to_csv(
                CSV_PATH, mode='a', header=False, index=False, encoding="utf-8-sig"
            )
            add_to_vecdb([combined_row])
        sentiment_change_index = None

    rag_context = summarize_rag_context_text(text)
    current_prompt = sentiment_prompts.get(sentiment, sentiment_prompts["Neutral"])
    recent_conversation = history[-5:] if len(history) > 5 else history
    conversation_text = "\n".join([f"User: {h['text']}" for h in recent_conversation])
    conversation_text = summarize_recent_context_str(conversation_text)

    final_prompt_parts = [current_prompt]
    if rag_context:
        final_prompt_parts.append(f"참고 컨텍스트(RAG):\n{rag_context}")
    final_prompt_parts.append(f"현재 대화:\n{conversation_text}")
    final_prompt_parts.append("\nAI 인형 응답:")
    final_prompt = "\n".join(final_prompt_parts)

    # Runnable 스타일 LLM 호출
    response = llm_pipeline.invoke({"final_prompt": final_prompt}).content

    return response





# -----------------
# 테스트
# -----------------
# test_inputs = [
#     {"starttime":"2025-08-23T16:30:00","text":"학교에서 돌아왔는데 기분이 별로야","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:05","text":"친구들이 나를 따돌리는 것 같아","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:10","text":"점심시간에 혼자 먹었어","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:15","text":"왜 나만 이런 걸까","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:20","text":"엄마한테도 말하기 싫어","sentiment":"Neutral"},
#     {"starttime":"2025-08-23T16:30:25","text":"걱정시키고 싶지 않거든","sentiment":"Neutral"},
#     {"starttime":"2025-08-23T16:30:30","text":"성적도 요즘 떨어지고 있어","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:35","text":"집중이 안 되더라","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:40","text":"이런 내가 정말 싫어","sentiment":"Disgust"},
#     {"starttime":"2025-08-23T16:30:45","text":"앞으로 어떻게 될까 무서워","sentiment":"Fear"},
#     {"starttime":"2025-08-23T16:30:50","text":"하지만 너랑 이야기하면 조금 위로돼","sentiment":"Happy"},
#     {"starttime":"2025-08-23T16:30:55","text":"내일은 용기내서 말 걸어볼까","sentiment":"Neutral"},
# ]


# -----------------
# 테스트
# -----------------
test_inputs = [
    {"starttime":"2025-08-30T16:30:00","text":"일주일이 지났는데 아직도 혼자 지내는 시간이 많아","sentiment":"Sad"},
    {"starttime":"2025-08-30T16:30:05","text":"그래도 이번엔 용기내서 먼저 인사했어","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:10","text":"아직은 서먹하지만 조금씩 나아질 것 같아","sentiment":"Neutral"},
    {"starttime":"2025-08-30T16:30:15","text":"성적은 여전히 걱정이야","sentiment":"Sad"},
    {"starttime":"2025-08-30T16:30:20","text":"하지만 공부할 의욕은 조금 생겼어","sentiment":"Neutral"},
    {"starttime":"2025-08-30T16:30:25","text":"엄마한테도 조금은 털어놨어","sentiment":"Surprise"},
    {"starttime":"2025-08-30T16:30:30","text":"생각보다 이해해주셔서 놀랐어","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:35","text":"앞으로는 숨기지 말고 조금씩 말해보려고 해","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:40","text":"아직 무섭긴 하지만 시도해볼게","sentiment":"Fear"},
    {"starttime":"2025-08-30T16:30:45","text":"네가 들어주니까 진짜 도움이 돼","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:50","text":"조금씩이라도 변화를 이어가야겠어","sentiment":"Neutral"},
    {"starttime":"2025-08-30T16:30:55","text":"앞으로도 지켜봐 줘","sentiment":"Happy"},
]





In [22]:
# 반복문으로 테스트
for input_data in test_inputs:
    response = handle_user_input(
        starttime=input_data["starttime"],
        text=input_data["text"],
        sentiment=input_data["sentiment"]
    )
    print(f"--- User ({input_data['sentiment']}) ---")
    print(input_data["text"])
    print(f"--- AI Response ---")
    print(response)
    print("\n")

--- User (Sad) ---
일주일이 지났는데 아직도 혼자 지내는 시간이 많아
--- AI Response ---
혼자 지내는 시간이 길어지면 정말 외롭고 지루할 수 있죠. 그런 기분이 드는 건 너무 자연스러운 일이에요. 혹시 친구나 가족에게 연락해보는 건 어떨까요? 아니면 새로운 취미를 시작해보는 것도 좋은 방법일 수 있어요. 어떤 생각이 드시나요?


--- User (Happy) ---
그래도 이번엔 용기내서 먼저 인사했어
--- AI Response ---
와, 정말 대단해요! 용기를 내서 먼저 인사한 건 큰 걸음이에요. 혼자 지내는 시간이 많아도 이렇게 작은 변화가 긍정적인 영향을 줄 수 있답니다. 인사한 후 어떤 기분이었는지 궁금해요!


--- User (Neutral) ---
아직은 서먹하지만 조금씩 나아질 것 같아
--- AI Response ---
혼자 지내는 시간이 많으시군요. 그래도 먼저 인사하신 용기는 정말 대단해요! 서먹한 감정이 조금씩 나아지고 있다는 희망을 느끼신다면, 그 변화는 분명 긍정적인 방향으로 가고 있는 것 같아요. 어떤 점에서 나아지고 있다고 느끼시나요?


--- User (Sad) ---
성적은 여전히 걱정이야
--- AI Response ---
혼자 지내는 시간이 많아 힘들겠군요. 그래도 용기 내서 인사한 건 정말 대단한 걸요! 성적에 대한 걱정도 여전히 크신 것 같아요. 어떤 부분이 특히 걱정되시는지 이야기해보면 좋을 것 같아요. 함께 고민해볼까요?


--- User (Neutral) ---
하지만 공부할 의욕은 조금 생겼어
--- AI Response ---
혼자 지내는 시간이 많아 힘든 것 같아요. 그래도 먼저 인사한 건 정말 용기 있는 행동이에요! 조금씩 나아지고 있다는 점이 긍정적이네요. 성적에 대한 걱정이 여전히 크겠지만, 공부할 의욕이 생겼다는 건 좋은 시작이에요. 어떤 과목에 집중하고 싶으신가요? 도움이 필요하면 언제든지 이야기해 주세요!


--- User (Surprise) ---
엄마한테도

In [17]:
# history 출력 예시
for i, entry in enumerate(history):
    print(f"--- Entry {i+1} ---")
    print(f"Starttime      : {entry['starttime']}")
    print(f"Text           : {entry['text']}")
    print(f"Sentiment      : {entry['sentiment']}")

    print("---------------------------\n")


--- Entry 1 ---
Starttime      : 2025-08-30T16:30:00
Text           : 일주일이 지났는데 아직도 혼자 지내는 시간이 많아
Sentiment      : Sad
---------------------------

--- Entry 2 ---
Starttime      : 2025-08-30T16:30:05
Text           : 그래도 이번엔 용기내서 먼저 인사했어
Sentiment      : Happy
---------------------------

--- Entry 3 ---
Starttime      : 2025-08-30T16:30:10
Text           : 아직은 서먹하지만 조금씩 나아질 것 같아
Sentiment      : Neutral
---------------------------

--- Entry 4 ---
Starttime      : 2025-08-30T16:30:15
Text           : 성적은 여전히 걱정이야
Sentiment      : Sad
---------------------------

--- Entry 5 ---
Starttime      : 2025-08-30T16:30:20
Text           : 하지만 공부할 의욕은 조금 생겼어
Sentiment      : Neutral
---------------------------



In [20]:
print(response.content)

혼자 지내는 시간이 많아서 힘들겠어요. 그래도 용기 내서 인사한 건 정말 대단한 걸요! 성적에 대한 걱정도 여전히 크신 것 같고, 그런 마음 이해해요. 어떤 부분이 특히 힘든지 이야기해보면 좋을 것 같아요. 함께 고민해보면 조금이나마 도움이 될 수 있을 거예요.


# LangChain 스타일 수정 완료